# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishigupgta1234-ux/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [10]:
import pandas as pd
import os

df = pd.read_csv('https://raw.githubusercontent.com/ishigupgta1234-ux/flyrank-ml-internship/refs/heads/main/data/raw/content_refresh_anonymized.csv')
print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

stale_check = df.groupby('freshness_tier').agg(
    avg_ctr=('ctr', 'mean'),
    avg_trend_pct=('trend_pct', 'mean'),
    n=('content_id', 'count')
)
print(stale_check)

                 avg_ctr  avg_trend_pct      n
freshness_tier                                
0-30            0.609021       0.784405  20480
181+            3.693276      -6.781203    174
31-90           0.117543      -7.373054    175
91-180          0.238367     -15.683224   9171


In [12]:
# --- signal 2: ctr vs position ---
# checking ctr actually drops off by position bucket the way we'd expect
pos_check = df.groupby('position_tier').agg(
    avg_ctr=('ctr', 'mean'),
    n=('content_id', 'count')
)
print(pos_check.sort_values('avg_ctr'))

                avg_ctr      n
position_tier                 
deep           0.150212   1319
page_3_5       0.222484   7242
striking       0.323239   7304
page_1         0.652467  11814
top_3          1.483611   2321


Signal 1 (staleness): [CONFIRMED/OPPOSITE/MIXED/FALSE] — [fill in after seeing the
printed table, e.g. "older freshness_tier buckets show lower avg_ctr and more negative
avg_trend_pct, n printed above"]

Signal 2 (ctr vs position): [CONFIRMED/OPPOSITE/MIXED/FALSE] — [fill in after seeing
the printed table]

So basically the idea is: a page is worth flagging if either it hasn't been touched in
a while (days_since_last_update is high, so whatever's on it is probably outdated) or
its CTR is bad compared to other pages in the same position bucket. Like if you're in
the "striking" bucket but getting "deep" bucket CTR, something's off with your title/
snippet, not your ranking.

Nothing here uses future data or anything derived from the outcome — just
days_since_last_update, ctr, position_tier, and search_volume, all of which exist on
the page right now.

Reason codes:
- STALE_CRAWL -> hasn't been updated in over 30 days
- CTR_UNDERPERFORM -> ctr well below the average for its position_tier
- STALE_AND_CTR -> both at once, strongest case
- NONE -> didn't trigger anything

Mapped to actions: REFRESH, FIX_SNIPPET_OR_TITLE, PRIORITY_REVIEW, NO_ACTION

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the ranked queue

STALE_DAYS_CUTOFF = 30
CTR_UNDERPERFORM_RATIO = 0.6

bucket_avg_ctr = df.groupby('position_tier')['ctr'].transform('mean')
is_stale = df['days_since_last_update'] > STALE_DAYS_CUTOFF
is_ctr_bad = df['ctr'] < (bucket_avg_ctr * CTR_UNDERPERFORM_RATIO)

def get_reason(s, c):
    if s and c:
        return 'STALE_AND_CTR'
    elif s:
        return 'STALE_CRAWL'
    elif c:
        return 'CTR_UNDERPERFORM'
    return 'NONE'

df['reason_code'] = [get_reason(s, c) for s, c in zip(is_stale, is_ctr_bad)]

action_map = {
    'STALE_AND_CTR': 'PRIORITY_REVIEW',
    'STALE_CRAWL': 'REFRESH',
    'CTR_UNDERPERFORM': 'FIX_SNIPPET_OR_TITLE',
    'NONE': 'NO_ACTION'
}
df['action'] = df['reason_code'].map(action_map)

df['staleness_score'] = (df['days_since_last_update'] / STALE_DAYS_CUTOFF).clip(upper=3)
df['ctr_gap_score'] = ((bucket_avg_ctr - df['ctr']) / bucket_avg_ctr).clip(lower=0)
df['volume_weight'] = df['search_volume'].rank(pct=True)
df['score'] = (df['staleness_score'] + df['ctr_gap_score']) * (0.5 + df['volume_weight'])

ranked = df[df['reason_code'] != 'NONE'].sort_values('score', ascending=False).reset_index(drop=True)

os.makedirs('work/outputs', exist_ok=True)
ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"n flagged: {len(ranked)} / {len(df)} total")
ranked[['content_id', 'reason_code', 'action', 'score']].head(10)

n flagged: 24532 / 30000 total


,content_id,reason_code,action,score
0,content_5ec29ae79c60,STALE_AND_CTR,PRIORITY_REVIEW,5.999637
1,content_454cc6654c6e,STALE_AND_CTR,PRIORITY_REVIEW,5.999637
2,content_deb54e9e19cd,STALE_AND_CTR,PRIORITY_REVIEW,5.999637
3,content_bf67a444faef,STALE_AND_CTR,PRIORITY_REVIEW,5.999637
4,content_84fe9d0a707a,STALE_AND_CTR,PRIORITY_REVIEW,5.998475
5,content_6b41450ae50c,STALE_AND_CTR,PRIORITY_REVIEW,5.997312
6,content_e7eb94e121b9,STALE_AND_CTR,PRIORITY_REVIEW,5.996368
7,content_201a4a56f4d6,STALE_AND_CTR,PRIORITY_REVIEW,5.996368
8,content_be12f5f1f683,STALE_AND_CTR,PRIORITY_REVIEW,5.996368
9,content_b40e32d5df10,STALE_AND_CTR,PRIORITY_REVIEW,5.996368


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Top-20 review

top20 = ranked.head(20)

for i, row in top20.iterrows():
    conf = "high, both signals agree" if row['reason_code'] == 'STALE_AND_CTR' else "medium, one signal only"
    if 'STALE' in row['reason_code']:
        wrong_if = "if the page was actually updated recently and the timestamp just didn't refresh"
    else:
        wrong_if = "if this page is new and just hasn't built up impressions yet, so low ctr is noise not a real problem"
    print(f"{i+1}. {row['content_id']} | action: {row['action']} | reason: {row['reason_code']} | score: {row['score']:.2f}")
    print(f"   confidence: {conf}")
    print(f"   what would make this wrong: {wrong_if}")
    print()

1. content_5ec29ae79c60 | action: PRIORITY_REVIEW | reason: STALE_AND_CTR | score: 6.00
   confidence: high, both signals agree
   what would make this wrong: if the page was actually updated recently and the timestamp just didn't refresh

2. content_454cc6654c6e | action: PRIORITY_REVIEW | reason: STALE_AND_CTR | score: 6.00
   confidence: high, both signals agree
   what would make this wrong: if the page was actually updated recently and the timestamp just didn't refresh

3. content_deb54e9e19cd | action: PRIORITY_REVIEW | reason: STALE_AND_CTR | score: 6.00
   confidence: high, both signals agree
   what would make this wrong: if the page was actually updated recently and the timestamp just didn't refresh

4. content_bf67a444faef | action: PRIORITY_REVIEW | reason: STALE_AND_CTR | score: 6.00
   confidence: high, both signals agree
   what would make this wrong: if the page was actually updated recently and the timestamp just didn't refresh

5. content_84fe9d0a707a | action: PRIORI

Row 1 (content_xxxx) is flagged PRIORITY_REVIEW because it's both stale and
underperforming ctr for its position — the strongest case in the queue. It'd be wrong
if the page was recently redesigned but the crawl date just hasn't caught up yet.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak picks + leakage check

weak_picks = top20[(top20['search_volume'] < 20) | (top20['impressions_last_30d'] < 50)]
print(f"{len(weak_picks)} of the top 20 look shaky due to low volume:")
weak_picks[['content_id', 'reason_code', 'search_volume', 'impressions_last_30d', 'score']]

7 of the top 20 look shaky due to low volume:


,content_id,reason_code,search_volume,impressions_last_30d,score
6,content_e7eb94e121b9,STALE_AND_CTR,22200.0,47,5.996368
7,content_201a4a56f4d6,STALE_AND_CTR,22200.0,0,5.996368
13,content_3f7dbbd55f0c,STALE_AND_CTR,14800.0,2,5.993026
16,content_608540486d95,STALE_AND_CTR,8100.0,0,5.986997
17,content_f421fb2e10ea,STALE_AND_CTR,5400.0,6,5.979951
18,content_dfb3be68b7c7,STALE_AND_CTR,5400.0,0,5.979951
19,content_d18c47f775a7,STALE_AND_CTR,5400.0,24,5.979951


The rows flagged above look shaky because their score is driven by tiny sample sizes —
a page with 5 impressions and 1 click can swing ctr wildly without meaning anything real.

Leakage check: score inputs are only days_since_last_update, ctr, position_tier, and
search_volume — all columns that exist on the page today. trend_pct, trend_direction,
and the last_30d/prev_30d comparison columns were NOT used in scoring, since those
describe outcomes/trends rather than pre-existing signals.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.